In [52]:
!rm -rf /content/tinyYOLO

In [53]:
%cd /content

/content


In [54]:
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q

Cloning into '/content/tinyYOLO'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 136 (delta 62), reused 116 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 216.42 KiB | 3.28 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/tinyYOLO
  Preparing metadata (setup.py) ... done


In [66]:
!git pull


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.48 KiB | 1.48 MiB/s, done.
From https://github.com/ShMazumder/tinyYOLO
   6503f8d..9df84a0  main       -> origin/main
Updating 6503f8d..9df84a0
Fast-forward
 notebooks/10_full_evaluation.py | 200 +++++++++++++++++++++-------------------
 1 file changed, 106 insertions(+), 94 deletions(-)


In [21]:
import sys; sys.path.insert(0, '.')
from tinyYOLO.models import build_model
from tinyYOLO.utils.env import print_env_report
import torch

print_env_report()
model, info = build_model(task='det', variant='standard')
x = torch.randn(1, 3, 320, 320)
out = model(x)
print(f'Model: {info["total_params_M"]}M params, output: {[o.shape for o in out]}')
print('Installation verified ✓')

AttributeError: 'torch._C._CudaDeviceProperties' object has no attribute 'total_mem'

In [56]:
!ls

analysis  notebooks  report.md	       scripts	 tinyYOLO
configs   README.md  requirements.txt  setup.py  tinyYOLO.egg-info


In [49]:
!python scripts/benchmark_models.py

  TinyYOLO Environment Report
  Platform:    COLAB
  OS:          Linux
  Python:      3.12.13
  PyTorch:     2.10.0+cu128
  GPU:         Tesla T4
  GPU Memory:  14.6 GB
  GPU Count:   1
  FP16:        Yes
  Device:      cuda:0
  Batch Size:  32 (recommended)
  Workers:     2
  Data Dir:    /content/datasets

  Model: tinyYOLO-det (standard)
  Parameters: 0.23M

 ImgSz |  Params(M) |   GFLOPs |       ms |      FPS
----------------------------------------------------
   160 |       0.23 |     0.04 |    34.4 |    29.1
   224 |       0.23 |     0.07 |    19.8 |    50.6
   320 |       0.23 |     0.15 |    25.9 |    38.6
   416 |       0.23 |     0.25 |    40.3 |    24.8
   640 |       0.23 |     0.59 |    83.2 |    12.0

  Model: tinyYOLO-det (quantized)
  Parameters: 0.22M

 ImgSz |  Params(M) |   GFLOPs |       ms |      FPS
----------------------------------------------------
   160 |       0.22 |     0.04 |    22.7 |    44.0
   224 |       0.22 |     0.07 |    26.6 |    37.6
   320 |  

In [57]:
# Train detection model
!python scripts/train.py --task det --variant standard --imgsz 320 --quick

  TinyYOLO Environment Report
  Platform:    COLAB
  OS:          Linux
  Python:      3.12.13
  PyTorch:     2.10.0+cu128
  GPU:         Tesla T4
  GPU Memory:  14.6 GB
  GPU Count:   1
  FP16:        Yes
  Device:      cuda:0
  Batch Size:  32 (recommended)
  Workers:     2
  Data Dir:    /content/datasets

  Training: tinyYOLO-det-std-320
  Task: det | Variant: standard | ImgSz: 320
  Parameters: 0.23M
  GFLOPs:     0.15
  Device:     cuda:0
  Batch:      32
  Epochs:     5
  Data:       coco128.yaml

  Loading dataset...

WARNING ⚠️ Dataset 'coco128.yaml' images not found, missing path '/content/tinyYOLO/datasets/coco128/images/train2017'
Unzipping /content/tinyYOLO/datasets/coco128.zip to /content/tinyYOLO/datasets/coco128...: 100% ━━━━━━━━━━━━ 263/263 3.1Kfiles/s 0.1s
Dataset download success ✅ (0.5s), saved to /content/tinyYOLO/datasets

  Train images: /content/tinyYOLO/datasets/coco128/images/train2017
  Dataset: 128 images, 4 batches

   Epoch      Box      Cls      Obj    To

In [58]:
# 3. Full training on COCO128 (100 epochs, auto-detects GPU/batch/AMP)

!python scripts/train.py --task det --variant standard --imgsz 320 --epochs 100


  TinyYOLO Environment Report
  Platform:    COLAB
  OS:          Linux
  Python:      3.12.13
  PyTorch:     2.10.0+cu128
  GPU:         Tesla T4
  GPU Memory:  14.6 GB
  GPU Count:   1
  FP16:        Yes
  Device:      cuda:0
  Batch Size:  32 (recommended)
  Workers:     2
  Data Dir:    /content/datasets

  Training: tinyYOLO-det-std-320
  Task: det | Variant: standard | ImgSz: 320
  Parameters: 0.23M
  GFLOPs:     0.15
  Device:     cuda:0
  Batch:      32
  Epochs:     100
  Data:       coco128.yaml

  Loading dataset...
  Train images: /content/tinyYOLO/datasets/coco128/images/train2017
  Dataset: 128 images, 4 batches

   Epoch      Box      Cls      Obj    Total      P      R     F1   mAP50         LR   Time
  ----------------------------------------------------------------------------------------
     1/100   1.0923   0.0680   0.3775   2.6301  0.000  0.000  0.000  0.0000   0.001000  24.8s
     2/100   1.0823   0.0678   0.3669   2.5994  0.000  0.000  0.000  0.0000   0.000999  

In [59]:
# Already have standard results, so just:
!python scripts/train.py --task det --variant quantized --imgsz 320 --epochs 100


  TinyYOLO Environment Report
  Platform:    COLAB
  OS:          Linux
  Python:      3.12.13
  PyTorch:     2.10.0+cu128
  GPU:         Tesla T4
  GPU Memory:  14.6 GB
  GPU Count:   1
  FP16:        Yes
  Device:      cuda:0
  Batch Size:  32 (recommended)
  Workers:     2
  Data Dir:    /content/datasets

  Training: tinyYOLO-det-q-320
  Task: det | Variant: quantized | ImgSz: 320
  Parameters: 0.22M
  GFLOPs:     0.15
  Device:     cuda:0
  Batch:      32
  Epochs:     100
  Data:       coco128.yaml

  Loading dataset...
  Train images: /content/tinyYOLO/datasets/coco128/images/train2017
  Dataset: 128 images, 4 batches

   Epoch      Box      Cls      Obj    Total      P      R     F1   mAP50         LR   Time
  ----------------------------------------------------------------------------------------
     1/100   1.0979   0.0683   0.3640   2.6282  0.000  0.000  0.000  0.0000   0.001000  25.8s
     2/100   1.0783   0.0678   0.3612   2.5857  0.000  0.000  0.000  0.0000   0.000999  1

In [63]:
!python notebooks/09_metrics_report.py


Found 2 experiments:
  • tinyYOLO-det-q-320
  • tinyYOLO-det-std-320

  HYPERPARAMETER CONFIGURATION REPORT
  Experiment: tinyYOLO-det-q-320

  Model Architecture:
    Task:          det
    Variant:       quantized
    Parameters:    0.22M
    Input Size:    320×320

  Training Settings:
    Epochs:        100
    Batch Size:    32
    Device:        cuda:0
    AMP (FP16):    True
    Platform:      colab

  Loss Function:
    Type:          CIoU + BCE
    Box Weight:    2.0
    Cls Weight:    1.0
    Obj Weight:    1.0

  Optimizer:
    Type:          AdamW
    Learning Rate: 0.001
    Betas:         [0.9, 0.999]
    Weight Decay:  1e-4 (weights only, biases/BN excluded)

  BatchNorm (YOLO-Standard):
    eps:           0.001
    momentum:      0.03

  Scheduler:
    Type:          CosineAnnealingLR
    T_max:         100
    eta_min:       1e-05

  AUGMENTATION REPORT
    Resize:            320×320
    Color Jitter:
      Brightness:    0.4
      Contrast:      0.4
      Saturation: 

In [ ]:
!python notebooks/10_full_evaluation.py


  TinyYOLO Environment Report
  Platform:    COLAB
  OS:          Linux
  Python:      3.12.13
  PyTorch:     2.10.0+cu128
  GPU:         Tesla T4
  GPU Memory:  14.6 GB
  GPU Count:   1
  FP16:        Yes
  Device:      cuda:0
  Batch Size:  32 (recommended)
  Workers:     2
  Data Dir:    /content/datasets
  TinyYOLO Environment Report
  Platform:    COLAB
  OS:          Linux
  Python:      3.12.13
  PyTorch:     2.10.0+cu128
  GPU:         Tesla T4
  GPU Memory:  14.6 GB
  GPU Count:   1
  FP16:        Yes
  Device:      cuda:0
  Batch Size:  32 (recommended)
  Workers:     2
  Data Dir:    /content/datasets

  Training: tinyYOLO-det-std-320
  Task: det | Variant: standard | ImgSz: 320
  Parameters: 0.23M
  GFLOPs:     0.15
  Device:     cuda:0
  Batch:      32
  Epochs:     100
  Data:       coco128.yaml

  Loading dataset...
  Train images: /content/tinyYOLO/datasets/coco128/images/train2017
  Dataset: 128 images, 4 batches

   Epoch      Box      Cls      Obj    Total      P    